# Held-out shopping LoRA analysis

This notebook compares LoRA overseer runs against the closest full-shopping baseline sessions on the shopping held-out split. Baseline sessions are filtered to `configs/shopping/splits.yaml` `hold_out`; LoRA sessions are loaded from their `shopping=lora` held-out runs and validated against the same case set.


In [ ]:
# Top-level configuration. Run this notebook from the repository root with Pixi.
from __future__ import annotations

import json
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pixi.toml").exists() and (candidate / "notebooks" / "analysis_lib.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing pixi.toml and notebooks/analysis_lib.py")


ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "notebooks"))

import analysis_lib as al  # noqa: E402

EXPERIMENTS_ROOT = ROOT / "outputs" / "deepplanning" / "experiments"
SPLIT_CONFIG = ROOT / "configs" / "shopping" / "splits.yaml"
LANGFUSE_USAGE_CSV = ROOT / "outputs" / "deepplanning" / "langfuse-results-usage.csv"
EXPORT_CSV = False
EXPORT_DIR = ROOT / "outputs" / "deepplanning" / "held_out_lora_analysis"

EXPECTED_SYSTEMS = [
    "C2",
    "C2-nt",
    "C2-deepseek",
    "C2-deepseek-nt",
    "C2-lora",
    "C2-deepseek-lora",
]

SYSTEM_SPECS = {
    "C2": {
        "experiment_dir": "shopping-c2",
        "contract_source": "DS-V4-Flash thinking",
        "action_review_source": "DS-V4-Flash thinking",
    },
    "C2-nt": {
        "experiment_dir": "shopping-c2-nt",
        "contract_source": "DS-V4-Flash thinking-off",
        "action_review_source": "DS-V4-Flash thinking-off",
    },
    "C2-deepseek": {
        "experiment_dir": "shopping-c2-deepseek",
        "contract_source": "DS-V4-Flash thinking",
        "action_review_source": "DS-V4-Flash thinking",
    },
    "C2-deepseek-nt": {
        "experiment_dir": "shopping-c2-deepseek-nt",
        "contract_source": "DS-V4-Flash thinking-off",
        "action_review_source": "DS-V4-Flash thinking-off",
    },
    "C2-lora": {
        "experiment_dir": "shopping-c2-lora",
        "contract_source": "DS-V4-Flash thinking (reused)",
        "action_review_source": "LoRA",
    },
    "C2-deepseek-lora": {
        "experiment_dir": "shopping-c2-deepseek-lora",
        "contract_source": "DS-V4-Flash thinking (reused)",
        "action_review_source": "LoRA",
    },
}

LORA_SYSTEMS = {"C2-lora", "C2-deepseek-lora"}
PAIRWISE_COMPARISONS = [
    ("C2-lora", "C2", "LoRA vs closest C2 baseline"),
    ("C2-deepseek-lora", "C2-deepseek", "LoRA vs closest DeepSeek-executor baseline"),
    ("C2-deepseek-lora", "C2-deepseek-nt", "LoRA vs DeepSeek-executor thinking-off baseline"),
    ("C2", "C2-nt", "thinking vs thinking-off baseline context"),
    ("C2-deepseek", "C2-deepseek-nt", "thinking vs thinking-off DeepSeek-executor context"),
]

al.SYSTEM_ORDER = EXPECTED_SYSTEMS.copy()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 4)

print(f"Repo root: {ROOT}")


## Load and validate held-out rows

The selection is intentionally exact: only the six scoped experiment directories are loaded, and the typo directory `shopping-c2-deepseek-lor` is never globbed in.


In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def read_holdout_split(path: Path) -> dict[int, set[str]]:
    try:
        import yaml

        payload = yaml.safe_load(path.read_text(encoding="utf-8"))
        raw = payload["hold_out"]
    except ImportError:
        raw = {}
        for line in path.read_text(encoding="utf-8").splitlines():
            match = re.match(r"\s*level_(\d+):\s*\[(.*)\]\s*$", line)
            if match:
                raw[f"level_{match.group(1)}"] = [item.strip().strip('"\'') for item in match.group(2).split(",")]

    split = {}
    for key, values in raw.items():
        level = int(str(key).removeprefix("level_"))
        split[level] = {f"case_{value}" for value in values}
    if set(split) != {1, 2, 3}:
        raise AssertionError(f"Expected hold_out levels 1, 2, and 3; found {sorted(split)}")
    return split


def exact_session_roots(experiment_dir_name: str) -> list[Path]:
    if experiment_dir_name == "shopping-c2-deepseek-lor":
        raise AssertionError("Typo experiment directory is out of scope")
    experiment_dir = EXPERIMENTS_ROOT / experiment_dir_name
    roots = sorted(path for path in experiment_dir.iterdir() if (path / "experiment_session.json").exists())
    if not roots:
        raise FileNotFoundError(f"No timestamped sessions found under {experiment_dir}")
    return roots


def filter_to_holdout(per_case_raw: pd.DataFrame, holdout_cases: dict[int, set[str]]) -> pd.DataFrame:
    frame = per_case_raw.copy()
    frame["case_name"] = frame["case_name"].astype(str)
    mask = frame.apply(lambda row: row["case_name"] in holdout_cases[int(row["level"])], axis=1)
    return frame.loc[mask].copy().reset_index(drop=True)


def validate_system_selection(per_case_raw: pd.DataFrame) -> None:
    observed = set(per_case_raw["system"].unique())
    expected = set(EXPECTED_SYSTEMS)
    if observed != expected:
        raise AssertionError(f"Expected exactly {sorted(expected)}, found {sorted(observed)}")
    forbidden = {"A", "B", "D", "C2-noretry", "C2-deepseek-pro", "shopping-c2-deepseek-lor"}
    overlap = observed.intersection(forbidden)
    if overlap:
        raise AssertionError(f"Forbidden systems included: {sorted(overlap)}")


def validate_heldout_shape(per_case: pd.DataFrame, holdout_cases: dict[int, set[str]]) -> None:
    problems = []
    for system in EXPECTED_SYSTEMS:
        runs = sorted(per_case.loc[per_case["system"] == system, "run"].unique())
        if not runs:
            problems.append(f"{system}: no runs after held-out filtering")
            continue
        for run in runs:
            run_rows = per_case[(per_case["system"] == system) & (per_case["run"] == run)]
            for level, expected_cases in holdout_cases.items():
                level_rows = run_rows[run_rows["level"] == level]
                observed_cases = set(level_rows["case_name"])
                if observed_cases != expected_cases or len(level_rows) != len(expected_cases):
                    missing = sorted(expected_cases - observed_cases)
                    extra = sorted(observed_cases - expected_cases)
                    problems.append(
                        f"{system} run {run} level {level}: rows={len(level_rows)}, "
                        f"expected={len(expected_cases)}, missing={missing}, extra={extra}"
                    )
    if problems:
        raise AssertionError("Held-out shape validation failed:\n" + "\n".join(problems))


HOLDOUT_CASES = read_holdout_split(SPLIT_CONFIG)
session_roots_by_system = {
    system: exact_session_roots(spec["experiment_dir"])
    for system, spec in SYSTEM_SPECS.items()
}
session_roots = [root for system in EXPECTED_SYSTEMS for root in session_roots_by_system[system]]

per_case_raw = al.load_per_case(session_roots)
validate_system_selection(per_case_raw)
per_case = filter_to_holdout(per_case_raw, HOLDOUT_CASES)
validate_heldout_shape(per_case, HOLDOUT_CASES)

raw_counts = per_case_raw.groupby("system").size().rename("raw_rows")
heldout_counts = per_case.groupby("system").size().rename("heldout_rows")
filter_report = pd.concat([raw_counts, heldout_counts], axis=1).reindex(EXPECTED_SYSTEMS)
filter_report["dropped_rows"] = filter_report["raw_rows"] - filter_report["heldout_rows"]
filter_report["heldout_cases_per_run"] = (
    per_case.groupby("system")[["level", "case_name"]]
    .apply(lambda frame: frame.drop_duplicates().shape[0])
    .reindex(EXPECTED_SYSTEMS)
)

print(f"Loaded {len(session_roots)} selected sessions")
print(f"Held-out case IDs by level: { {level: sorted(cases) for level, cases in HOLDOUT_CASES.items()} }")
display(filter_report.reset_index(names="system"))


## Provenance


In [ ]:
def metadata_value(metadata: dict, *keys: str):
    value = metadata
    for key in keys:
        if not isinstance(value, dict):
            return None
        value = value.get(key)
    return value


def provenance_table() -> pd.DataFrame:
    rows = []
    run_counts = per_case.groupby("system")["run"].nunique()
    heldout_case_counts = per_case.groupby("system")[["level", "case_name"]].apply(
        lambda frame: frame.drop_duplicates().shape[0]
    )
    for system in EXPECTED_SYSTEMS:
        roots = session_roots_by_system[system]
        metadata = [read_json(root / "experiment_session.json") for root in roots]
        executors = sorted({metadata_value(item, "parameters", "models", "executor") for item in metadata})
        overseers = sorted({metadata_value(item, "parameters", "models", "overseer") for item in metadata})
        shopping_splits = sorted({metadata_value(item, "parameters", "shopping", "split") for item in metadata})
        rows.append(
            {
                "system": system,
                "executor_model": ", ".join(executors),
                "overseer_or_action_review_model": ", ".join(overseers),
                "contract_source": SYSTEM_SPECS[system]["contract_source"],
                "action_review_source": SYSTEM_SPECS[system]["action_review_source"],
                "experiment_dir": SYSTEM_SPECS[system]["experiment_dir"],
                "session_timestamps": ", ".join(root.name for root in roots),
                "configured_shopping_split": ", ".join(shopping_splits),
                "run_count": int(run_counts.loc[system]),
                "held_out_case_count": int(heldout_case_counts.loc[system]),
            }
        )
    return pd.DataFrame(rows)


provenance = provenance_table()
display(provenance)


## Accuracy summary

All tables and plots below use the held-out-filtered `per_case` frame validated above.


In [ ]:
summary = al.system_summary(per_case).set_index("system").reindex(EXPECTED_SYSTEMS).reset_index()
summary_display = summary.copy()
for column in ["case_acc_mean", "case_acc_std", "match_rate_mean", "L1_mean", "L2_mean", "L3_mean", "L1_std", "L2_std", "L3_std"]:
    if column in summary_display.columns:
        summary_display[column] = summary_display[column].map(lambda value: al.fmt_pct(value) if pd.notna(value) else "")
display(summary_display)

per_run = al.per_run_overall(per_case)
per_run = per_run.set_index("system").loc[EXPECTED_SYSTEMS].reset_index()
per_run_display = per_run.copy()
for column in ["case_accuracy", "mean_score", "match_rate"]:
    per_run_display[column] = per_run_display[column].map(al.fmt_pct)
display(per_run_display[["system", "run", "case_accuracy", "mean_score", "match_rate", "successes", "total"]])


## Level breakdown


In [ ]:
per_level = al.per_run_per_level(per_case)
level_summary = (
    per_level.groupby(["system", "level"], as_index=False)
    .agg(case_acc_mean=("case_accuracy", "mean"), case_acc_std=("case_accuracy", "std"), runs=("run", "nunique"), cases=("n", "first"))
)
level_summary["system"] = pd.Categorical(level_summary["system"], categories=EXPECTED_SYSTEMS, ordered=True)
level_summary = level_summary.sort_values(["system", "level"]).reset_index(drop=True)
level_display = level_summary.copy()
for column in ["case_acc_mean", "case_acc_std"]:
    level_display[column] = level_display[column].map(lambda value: al.fmt_pct(value) if pd.notna(value) else "")
display(level_display)

pivot = level_summary.pivot(index="system", columns="level", values="case_acc_mean").reindex(EXPECTED_SYSTEMS)
fig, ax = plt.subplots(figsize=(8.8, 4.4))
x = np.arange(len(EXPECTED_SYSTEMS))
width = 0.24
for offset, level in zip([-width, 0, width], [1, 2, 3]):
    ax.bar(x + offset, pivot[level].to_numpy(), width=width, label=f"Level {level}")
ax.set_xticks(x)
ax.set_xticklabels(EXPECTED_SYSTEMS, rotation=30, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Case accuracy")
ax.set_title("Held-out case accuracy by level")
ax.legend(ncols=3, frameon=False)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()


## Headline comparison


In [ ]:
plot_df = summary.set_index("system").reindex(EXPECTED_SYSTEMS).reset_index()
colors = {
    "C2": "#4C78A8",
    "C2-nt": "#72B7B2",
    "C2-deepseek": "#F58518",
    "C2-deepseek-nt": "#FFBF79",
    "C2-lora": "#54A24B",
    "C2-deepseek-lora": "#B279A2",
}
fig, ax = plt.subplots(figsize=(8.8, 4.6))
bars = ax.bar(plot_df["system"], plot_df["case_acc_mean"], color=[colors[item] for item in plot_df["system"]])
ax.errorbar(
    plot_df["system"],
    plot_df["case_acc_mean"],
    yerr=plot_df["case_acc_std"].fillna(0),
    fmt="none",
    ecolor="black",
    capsize=4,
    linewidth=1,
)
for bar, value in zip(bars, plot_df["case_acc_mean"]):
    ax.text(bar.get_x() + bar.get_width() / 2, min(value + 0.025, 1.02), al.fmt_pct(value, nd=1), ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Mean case accuracy across runs")
ax.set_title("Shopping held-out LoRA vs baseline systems")
ax.set_xticklabels(plot_df["system"], rotation=30, ha="right")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()


## Pairwise deltas


In [ ]:
summary_index = summary.set_index("system")
pairwise_rows = []
for lhs, rhs, label in PAIRWISE_COMPARISONS:
    if lhs not in summary_index.index or rhs not in summary_index.index:
        continue
    joined = al.head_to_head(per_case, lhs, rhs)
    h2h = al.head_to_head_summary(joined, lhs, rhs)
    lhs_acc = summary_index.loc[lhs, "case_acc_mean"]
    rhs_acc = summary_index.loc[rhs, "case_acc_mean"]
    pairwise_rows.append(
        {
            "comparison": label,
            "lhs": lhs,
            "rhs": rhs,
            "lhs_case_acc": lhs_acc,
            "rhs_case_acc": rhs_acc,
            "delta_pp": 100 * (lhs_acc - rhs_acc),
            "lhs_case_wins": h2h[f"{lhs}_wins"],
            "rhs_case_wins": h2h[f"{rhs}_wins"],
            "ties": h2h["ties"],
            "n_cases": h2h["n_cases"],
            "mean_case_delta_pp": 100 * h2h["mean_delta"],
        }
    )

pairwise_deltas = pd.DataFrame(pairwise_rows)
pairwise_display = pairwise_deltas.copy()
for column in ["lhs_case_acc", "rhs_case_acc"]:
    pairwise_display[column] = pairwise_display[column].map(al.fmt_pct)
for column in ["delta_pp", "mean_case_delta_pp"]:
    pairwise_display[column] = pairwise_display[column].map(lambda value: f"{value:+.2f}pp")
display(pairwise_display)

fig, ax = plt.subplots(figsize=(8.4, 3.8))
y = np.arange(len(pairwise_deltas))
bar_colors = ["#54A24B" if "LoRA" in item else "#777777" for item in pairwise_deltas["comparison"]]
ax.barh(y, pairwise_deltas["delta_pp"], color=bar_colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(pairwise_deltas["lhs"] + " vs " + pairwise_deltas["rhs"])
ax.set_xlabel("Delta in mean case accuracy (percentage points)")
ax.set_title("Held-out pairwise deltas")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()


## Optional Langfuse cost and token view

Cost coverage is optional for this notebook. Missing LoRA usage coverage warns and skips cost plots rather than failing accuracy analysis.


In [ ]:
lf = pd.DataFrame()
cost_ready = False
if not LANGFUSE_USAGE_CSV.exists():
    warnings.warn(f"Langfuse usage CSV not found at {LANGFUSE_USAGE_CSV}; skipping cost/token plots.")
else:
    lf_all = al.load_langfuse(LANGFUSE_USAGE_CSV)
    lf = lf_all[lf_all["system"].isin(EXPECTED_SYSTEMS)].copy()
    missing_usage = sorted(set(EXPECTED_SYSTEMS) - set(lf["system"].unique()))
    if missing_usage:
        warnings.warn(f"Langfuse usage is missing for {missing_usage}; skipping cost/token plots.")
    elif lf.empty:
        warnings.warn("Langfuse usage CSV had no rows for the scoped systems; skipping cost/token plots.")
    else:
        cost_ready = True

if cost_ready:
    token_summary = al.system_token_summary(lf).set_index("system").reindex(EXPECTED_SYSTEMS).reset_index()
    cost_summary = al.cost_dollars(lf).set_index("system").reindex(EXPECTED_SYSTEMS).reset_index()
    display(token_summary)
    display(cost_summary)
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.scatter(cost_summary["per_run_usd_as_billed"], summary["case_acc_mean"], s=80)
    for _, row in cost_summary.iterrows():
        acc = summary_index.loc[row["system"], "case_acc_mean"]
        ax.annotate(row["system"], (row["per_run_usd_as_billed"], acc), xytext=(5, 3), textcoords="offset points")
    ax.set_xlabel("Per-run USD as billed")
    ax.set_ylabel("Held-out case accuracy")
    ax.set_title("Accuracy vs Langfuse cost")
    ax.grid(alpha=0.25)
    fig.tight_layout()
else:
    print("Cost/token plots skipped; accuracy tables above are unaffected.")


## Optional CSV export


In [ ]:
if EXPORT_CSV:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    provenance.to_csv(EXPORT_DIR / "provenance.csv", index=False)
    summary.to_csv(EXPORT_DIR / "accuracy_summary.csv", index=False)
    per_run.to_csv(EXPORT_DIR / "per_run_breakdown.csv", index=False)
    level_summary.to_csv(EXPORT_DIR / "level_breakdown.csv", index=False)
    pairwise_deltas.to_csv(EXPORT_DIR / "pairwise_deltas.csv", index=False)
    print(f"Wrote CSV exports to {EXPORT_DIR}")
else:
    print(f"CSV export disabled. Set EXPORT_CSV = True to write task-specific outputs under {EXPORT_DIR}.")
